# Explore Character Graphs

Quick look at the graphs produced by the pipeline:

1. Chapter graphs from BookNLP (`data/graphs/chapters/<slug>.pkl`).
2. The LitBank gold mini-graph for the same book (`data/graphs/gold/<slug>_gold.pkl`).

We will:

1. Load both for one book.
2. Print per-chapter stats (nodes, edges, density).
3. Draw chapter 1 and chapter 5 side-by-side.
4. Peek at the strongest edges + their sentiment.
5. Compare the BookNLP graph for chapter 1 against the LitBank gold graph.

Make sure you have already run the pipeline, e.g.:

```
python -m src.pipeline --book 1342
```


In [ ]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Make sure we can import the src package whether the notebook is launched
# from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

CHAPTERS_DIR = PROJECT_ROOT / "data" / "graphs" / "chapters"
GOLD_DIR = PROJECT_ROOT / "data" / "graphs" / "gold"

available = sorted(p.stem for p in CHAPTERS_DIR.glob("*.pkl"))
print("Available books (chapter pickles):", available)


In [ ]:
# Pick which book to inspect. Options after the smoke run include:
#   "bleak_house", "persuasion", "the_masque_of_the_red_death".
BOOK_SLUG = "persuasion"

with open(CHAPTERS_DIR / f"{BOOK_SLUG}.pkl", "rb") as f:
    graphs = pickle.load(f)

gold_path = GOLD_DIR / f"{BOOK_SLUG}_gold.pkl"
gold_graph = None
if gold_path.exists():
    with open(gold_path, "rb") as f:
        gold_graph = pickle.load(f)

print(f"Loaded {len(graphs)} chapter graphs for '{BOOK_SLUG}'.")
if gold_graph is not None:
    print(f"Gold graph: {gold_graph.number_of_nodes()} nodes, "
          f"{gold_graph.number_of_edges()} edges.")


## Per-chapter stats

In [ ]:
rows = []
for g in graphs:
    rows.append({
        "chapter_id": g.graph.get("chapter_id"),
        "chapter_title": g.graph.get("chapter_title", ""),
        "num_tokens": g.graph.get("num_tokens", 0),
        "short": g.graph.get("short_chapter", False),
        "nodes": g.number_of_nodes(),
        "edges": g.number_of_edges(),
        "density": round(nx.density(g), 4) if g.number_of_nodes() > 1 else 0.0,
    })

stats = pd.DataFrame(rows)
stats


## Visualize chapter 1 and chapter 5

In [ ]:
SENTIMENT_COLOR = {"positive": "#2ca02c", "negative": "#d62728", "neutral": "#7f7f7f"}


def draw_graph(graph: nx.Graph, ax, title: str, *, use_sentiment: bool = True) -> None:
    if graph.number_of_nodes() == 0:
        ax.set_title(f"{title}\n(empty)")
        ax.axis("off")
        return

    pos = nx.spring_layout(graph, seed=42, k=0.8)

    sizes = [300 + 40 * graph.nodes[n].get("mention_count", 1) for n in graph.nodes]
    labels = {n: graph.nodes[n].get("name", str(n)) for n in graph.nodes}

    if use_sentiment:
        edge_colors = [SENTIMENT_COLOR.get(graph.edges[e].get("sentiment", "neutral"), "#7f7f7f")
                       for e in graph.edges]
    else:
        edge_colors = ["#555555"] * graph.number_of_edges()
    edge_widths = [0.6 + 0.3 * graph.edges[e].get("weight", 1) for e in graph.edges]

    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color=edge_colors, width=edge_widths, alpha=0.7)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_size=sizes, node_color="#1f77b4", alpha=0.85)
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=8, ax=ax)

    ax.set_title(title)
    ax.axis("off")


def pick_by_chapter_id(graphs, target_id: int):
    for g in graphs:
        if g.graph.get("chapter_id") == target_id:
            return g
    return graphs[target_id] if target_id < len(graphs) else None


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, chap_id in zip(axes, [0, 4]):  # chapter 1 = index 0, chapter 5 = index 4
    g = pick_by_chapter_id(graphs, chap_id)
    if g is None:
        ax.set_title(f"Chapter {chap_id + 1} not available")
        ax.axis("off")
        continue
    draw_graph(g, ax, title=g.graph.get("chapter_title", f"Chapter {chap_id + 1}"))

plt.tight_layout()
plt.show()


## Peek at a few edges with descriptions

Handy sanity-check: do the edge "description" snippets actually make sense?

In [ ]:
sample = pick_by_chapter_id(graphs, 0)
if sample is not None:
    top_edges = sorted(sample.edges(data=True), key=lambda e: -e[2].get("weight", 0))[:5]
    for u, v, data in top_edges:
        print(f"{sample.nodes[u]['name']}  <->  {sample.nodes[v]['name']}  "
              f"[weight={data['weight']}, sentiment={data['sentiment']}]")
        print(f"    {data['description']}")
        print()


## BookNLP chapter 1 vs LitBank gold

LitBank annotations cover only ~2 000 words — typically the opening of the
book — so the natural comparison is BookNLP on chapter 1 vs the gold
mini-graph.

In [ ]:
if gold_graph is None:
    print("No gold graph available for this book.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    draw_graph(pick_by_chapter_id(graphs, 0), axes[0], title="BookNLP — Chapter 1")
    draw_graph(gold_graph, axes[1], title="LitBank gold (first ~2000 words)",
               use_sentiment=False)
    plt.tight_layout()
    plt.show()

    booknlp_names = {d["name"].lower() for _, d in pick_by_chapter_id(graphs, 0).nodes(data=True)}
    gold_names = {d["name"].lower() for _, d in gold_graph.nodes(data=True)}
    overlap = booknlp_names & gold_names
    only_booknlp = booknlp_names - gold_names
    only_gold = gold_names - booknlp_names
    print(f"Shared character names  ({len(overlap):>3}): {sorted(overlap)[:10]} ...")
    print(f"Only in BookNLP chap. 1 ({len(only_booknlp):>3}): {sorted(only_booknlp)[:10]} ...")
    print(f"Only in LitBank gold    ({len(only_gold):>3}): {sorted(only_gold)[:10]} ...")
